Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import pandas as pd

query = """
WITH base_table AS (
    SELECT
        x.`id` AS id,
        x.`policy_id` AS policy_id,
        x.`claim_type` AS claim_type,
        x.`patient_name` AS patient_name,
        x.`hospital_name` AS hospital_name,
        COALESCE(x.hospital_pincode, gh.pincode) AS hospital_pincode,
        x.`hospital_address` AS hospital_address,
        x.`admission_date` AS admission_date,
        x.`date_of_discharge` AS date_of_discharge,
        x.`icd_code` AS icd_code,
        x.`treatment` AS treatment,
        x.`claimed_amount` AS claimed_amount,
        x.`approved_amount` AS approved_amount,
        x.`claim_id` AS claim_id,
        x.`claim_last_date_trail_new` AS claim_last_date_trail_new,
        x.`status_ct` AS status_ct,
        x.`type_of_treatment` AS type_of_treatment,
        c.done_by AS me_name,
        c.from_status,
        c.to_status,
        to_timestamp(c.audit_time) AS audit_time,
        CAST(to_date(to_timestamp(c.audit_time)) AS DATE) AS dt,
        CASE
            WHEN c.from_status = 'PENDING_DATA_ENTRY_ME'
             AND c.to_status = 'PENDING_ADJUDICATION'
             AND x.`claim_type` = 'PRE_POST_HOSPITALISATION'
            THEN 'PRE_POST_HOSPITALISATION'
            WHEN c.from_status = 'PENDING_DATA_ENTRY_ME'
             AND c.to_status = 'PENDING_ADJUDICATION'
             AND x.`claim_type` = 'CASHLESS'
            THEN 'CASHLESS'
            WHEN c.from_status = 'PENDING_DATA_ENTRY_ME'
             AND c.to_status = 'PENDING_ADJUDICATION'
             AND x.`claim_type` = 'REIMBURSEMENT'
            THEN 'REIMBURSEMENT'
            WHEN c.from_status = 'PENDING_REJECTION_LETTER_CREATION'
             AND c.to_status = 'PENDING_REJECTION_LETTER_REVIEW'
            THEN 'PENDING_REJECTION_CREATION_TO_REVIEW'
        END AS action_category
    FROM gi_claims.claim_audit_trail c
    JOIN corp_analytics.ss_gi_claims_all_tbl x
        ON c.claim_id = x.`id`
    LEFT JOIN (
        SELECT hospital_name AS h_name, pincode
        FROM gi_partner.hospital
    ) gh
        ON gh.h_name = x.`hospital_name`
    WHERE (
            (c.from_status = 'PENDING_DATA_ENTRY_ME' AND c.to_status = 'PENDING_ADJUDICATION')
         OR (c.from_status = 'PENDING_REJECTION_LETTER_CREATION' AND c.to_status = 'PENDING_REJECTION_LETTER_REVIEW')
        )
      AND x.`year_week_crd` NOT IN ('Null', 'null')
),

base_date AS (
    SELECT
        bt.*,
        ROW_NUMBER() OVER (
            PARTITION BY id, audit_time, from_status, to_status, me_name
            ORDER BY claim_last_date_trail_new DESC
        ) AS rk
    FROM base_table bt
)

SELECT
    id,
    policy_id,
    claim_type,
    patient_name,
    hospital_name,
    hospital_pincode,
    hospital_address,
    admission_date,
    date_of_discharge,
    icd_code,
    treatment,
    claimed_amount,
    approved_amount,
    claim_last_date_trail_new,
    dt,
    status_ct,
    type_of_treatment,
    me_name
FROM base_date
WHERE rk = 1
  AND dt BETWEEN date_sub(current_date(), 2) AND date_sub(current_date(), 1)
  AND me_name IS NOT NULL
ORDER BY me_name, audit_time, id
"""

df = get_sql_df(query)

print("Rows Returned:", df.shape[0])
print("Columns Returned:", df.shape[1])

if not df.empty:
    display(df)
else:
    print("No rows returned.")

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1bHauzMgM8gHR6nfwDWUm8ngYRasHMN85gj65Wt1vDIc/edit?gid=0#gid=0"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)